In [1]:
!pip install opencv-python torch torchvision matplotlib scikit-learn pandas gradio

In [2]:
import os
os.makedirs("models", exist_ok=True)
os.makedirs("dataset/train", exist_ok=True)
os.makedirs("dataset/val", exist_ok=True)

In [3]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

def extract_palette(image_path, k=5):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError("Image not found or invalid")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    pixels = img.reshape(-1, 3).astype(np.float32)

    # K-means clustering
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(pixels)
    centers = kmeans.cluster_centers_.astype(int)

    # Convert RGB -> HEX
    palette_hex = [f"#{r:02x}{g:02x}{b:02x}" for r, g, b in centers]
    return palette_hex

In [4]:
from google.colab import files
uploaded = files.upload()   # Upload a room photo
filepath = list(uploaded.keys())[0]
palette = extract_palette(filepath)
print("Palette HEX:", palette)

Saving 3d-rendering-loft-luxury-living-room-with-bookshelf-near-bookshelf_105762-2224.avif to 3d-rendering-loft-luxury-living-room-with-bookshelf-near-bookshelf_105762-2224.avif
Palette HEX: ['#beb19d', '#68533e', '#dedbd4', '#968670', '#231e18']


In [5]:
# Create dummy dataset (replace with real images later)
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import random

class DummyRoomDataset(Dataset):
    def __init__(self, num_samples=200, img_size=224):
        self.num_samples = num_samples
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        self.classes = ['Modern', 'Minimalist', 'Rustic', 'Classic']
        self.class_to_idx = {c:i for i,c in enumerate(self.classes)}

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Create a random colored image (just for demo)
        img = Image.new('RGB', (224, 224), (random.randint(0,255), random.randint(0,255), random.randint(0,255)))
        label = random.randint(0, 3)
        return self.transform(img), label

train_dataset = DummyRoomDataset(150)
val_dataset = DummyRoomDataset(50)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

In [6]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.mobilenet_v2(pretrained=True)
num_classes = 4   # modern, minimalist, rustic, classic

# Replace classifier
model.classifier = nn.Sequential(
    nn.Dropout(0.2),
    nn.Linear(model.last_channel, 128),
    nn.ReLU(),
    nn.Linear(128, num_classes)
)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop (2 epochs for demo)
for epoch in range(2):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader):.4f}")

# Save model
torch.save(model.state_dict(), "models/style_model.pth")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 116MB/s] 


Epoch 1, Loss: 1.4821
Epoch 2, Loss: 1.6043


In [8]:
def predict_style(image_path):
    from PIL import Image
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    img = Image.open(image_path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = torch.max(outputs, 1)
    class_names = ['Modern', 'Minimalist', 'Rustic', 'Classic']
    return class_names[predicted.item()]

In [9]:
style = predict_style(filepath)
print("Predicted style:", style)

Predicted style: Minimalist


In [10]:
recommendations = {
    "Modern": {
        "furniture": "Clean lines, polished metal/glass, neutral upholstery",
        "layout": "Open floor plan, minimal clutter, statement art piece",
        "lighting": "Track lighting, large windows, LED strips"
    },
    "Minimalist": {
        "furniture": "Functional pieces, hidden storage, monochrome tones",
        "layout": "Less is more, negative space, simple shapes",
        "lighting": "Recessed lights, natural light priority"
    },
    "Rustic": {
        "furniture": "Wooden beams, reclaimed wood tables, leather sofa",
        "layout": "Cozy, central fireplace, warm textiles",
        "lighting": "Warm bulbs, wrought iron chandeliers"
    },
    "Classic": {
        "furniture": "Tufted sofas, dark wood, elegant details",
        "layout": "Symmetrical arrangements, formal dining area",
        "lighting": "Crystal chandeliers, sconces"
    }
}

def get_recommendations(style, palette):
    rec = recommendations.get(style, recommendations["Modern"])
    # Simple color-based refinement (optional)
    # e.g., if palette contains warm browns -> emphasize wood
    return rec

In [11]:
!pip install gradio

In [12]:
import gradio as gr

def interior_advisor(image):
    # Save uploaded image temporarily
    temp_path = "temp.jpg"
    image.save(temp_path)

    # Extract palette
    palette = extract_palette(temp_path)
    # Predict style
    style = predict_style(temp_path)
    # Get recommendations
    rec = get_recommendations(style, palette)

    # Format output
    palette_html = "<div style='display:flex; gap:10px'>" + "".join([f"<div style='background:{c}; width:50px; height:50px; border-radius:5px; border:1px solid #ccc'></div>" for c in palette]) + "</div>"

    result = f"""
    ## 🎨 Detected Style: **{style}**

    ### 🖌️ Color Palette:
    {palette_html}

    ### 🛋️ Furniture Recommendations:
    {rec['furniture']}

    ### 📐 Layout Advice:
    {rec['layout']}

    ### 💡 Lighting Tips:
    {rec['lighting']}
    """
    return result

iface = gr.Interface(
    fn=interior_advisor,
    inputs=gr.Image(type="pil"),
    outputs=gr.Markdown(),
    title="Interior Design Advisor",
    description="Upload a room photo and get AI-powered design recommendations."
)
iface.launch(share=True)   # Gives a public link

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://71d6efa93b510cf274.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
